In [1]:
import pandas as pd
from pathlib import Path
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import pandas as pd
import numpy as np
import torch
from collections import Counter
import re
import nltk
# import matplotlib.pyplot as plt
# import seaborn as sns

In [ ]:
train_path = Path.cwd().parent / "data/train.csv"
test_path = Path.cwd().parent / "data/test.csv"

In [13]:
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(train['text'], train['generated'], test_size=0.1, random_state=42)
valid = pd.concat([X_test, y_test], axis=1)
train = pd.concat([X_train, y_train], axis=1)

In [15]:
print(f"Среднее значение таргета = {train['generated'].mean()}")
print(f"Размер тренировочной выборки = {train['generated'].shape[0]}")

Среднее значение таргета = 0.3720086998908238
Размер тренировочной выборки = 350809


In [16]:
train['text'].iloc[1102]

"All over the world, Major cities are trying to reduce their use of cars. From Germany, to France, to Columbia, and coming back to the US, people are lqZqtqng car use to reduce pollution, or love better, or even to save Money. In some places, there QS one day every year that doesn't allow cars. In other places, there are whole coZZunqtqes that don't allow cars year round.\n\nIn Germany, there QS a community in named Vauban that doesn't allow cars unless you have the Money to pay for a parking spot that QS on the outskirts of town. The revolutionizing community Makes people pay $40, 000 dollars just for one parking spot. This wonderful new place has reduced the amount of greenhouse gas produced in the last few years drastically, with 70% of the population not owning a car. This has encouraged Zany other countries to start something just like qt.\n\nThe city of love Day not be so lovely QF qt's so soggy you can barely see. Ears was gaining so Such AQR pollution from the amount of gas tha

In [ ]:
def advanced_preprocessing(text, remove_stopwords=True, lemmatize=True):    
    tokens = nltk.word_tokenize(text)
    
    return ' '.join(tokens)

In [ ]:
from tqdm import tqdm
tqdm.pandas()

train['processed_text'] = train['text'].progress_apply(lambda text: ' '.join(nltk.word_tokenize(text)))

100%|██████████| 389788/389788 [09:57<00:00, 652.76it/s]


In [ ]:
valid['processed_text'] = valid['text'].progress_apply(lambda text: ' '.join(nltk.word_tokenize(text)))
test['processed_text'] = test['text'].progress_apply(lambda text: ' '.join(nltk.word_tokenize(text)))

valid.to_csv("valid_tokenized.csv")
test.to_csv("test_tokenized.csv")

100%|██████████| 97447/97447 [02:28<00:00, 656.41it/s]


In [9]:
train.head()

NameError: name 'train' is not defined

In [9]:
train[['text', 'processed_text']].head()

,text,processed_text
0,"I think that FACS is very useful technology, t...","I think that FACS is very useful technology , ..."
1,Should students create their own summer projec...,Should students create their own summer projec...
2,"As an average 8thgrade student, I have develop...","As an average 8thgrade student , I have develo..."
3,Holy Avocados! A new computer software has jus...,Holy Avocados ! A new computer software has ju...
4,Title: A Cowboy Who Rode the Waves\n\nOnce upo...,Title : A Cowboy Who Rode the Waves Once upon ...


In [2]:
import yaml
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, Any

@dataclass
class DataConfig:
    data_train: str
    data_valid: str
    data_test: str
    
@dataclass
class TrainingConfig:
    batch_size: int
    num_workers: int
    random_seed: int
    max_epochs: int
    learning_rate: float
    
@dataclass
class ModelConfig:
    embedding_dim: int
    hidden_dim: int
    num_layers: int
    dropout: float
    bidirectional: bool

class ConfigLoader:
    def __init__(self, config_dir: str = "configs"):
        self.config_dir = Path.cwd().parent / config_dir
        
    def load_yaml(self, filename: str) -> Dict[str, Any]:
        """Загрузка YAML файла"""
        filepath = self.config_dir / filename

        if not filepath.exists():
            raise FileNotFoundError(f"Config file not found: {filepath}")
            
        with open(filepath, 'r', encoding='utf-8') as f:
            return yaml.safe_load(f)
    
    def load_data_config(self) -> DataConfig:
        config_dict = self.load_yaml("data_load.yaml")
        return DataConfig(**config_dict)
    
    def load_training_config(self) -> TrainingConfig:
        config_dict = self.load_yaml("training.yaml")
        return TrainingConfig(**config_dict)
    
    def load_model_config(self) -> ModelConfig:
        config_dict = self.load_yaml("model.yaml")
        return ModelConfig(**config_dict)


loader = ConfigLoader("configs")

data_config = loader.load_data_config()
training_config = loader.load_training_config()
model_config = loader.load_model_config()

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from dev.dataset import create_dataloaders

In [5]:
train_loader, valid_loader, test_loader = create_dataloaders(data_config, training_config, model_config)

Vocabulary built with 128 tokens
Top 10 tokens: ['<PAD>', '<UNK>', '<SOS>', '<EOS>', 'the', 'to', 'and', 'a', 'of', 'in']


In [6]:
from dev.model import GRUClassifier

In [7]:
model = GRUClassifier(
            vocab_size=5000,
            model_config=model_config)

# Тестовый forward pass
batch_size = 4
seq_len = 32
test_input = torch.randint(0, 5000, (batch_size, seq_len))
test_lengths = torch.randint(20, seq_len, (batch_size,))

with torch.no_grad():
    output = model(test_input, test_lengths)

In [8]:
from dev.train import train_model

c:\ai_human_text_classification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
# train_model(
#     model=model,
#     train_loader=train_loader,
#     val_loader=valid_loader,
#     training_config=training_config,
#     model_config=model_config,
#     mlflow_config = None,
#     epochs=2,
#     device = None
# )